<a href="https://colab.research.google.com/github/manoharreddyvoladri/Quantum_Classification/blob/main/i_like_maize_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

manoharreddyvoladri_project_2_path = kagglehub.dataset_download('manoharreddyvoladri/project-2')

print('Data source import complete.')


In [ ]:
import os
import json
import torch
import torch.utils.data as data
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms as transforms
from PIL import Image, ImageDraw
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Improved Dataset Class with clear label mapping
class MaizeWeedDataset(data.Dataset):
    def __init__(self, root_dir, transform=None, use_json=True):
        self.root_dir = root_dir
        self.transform = transform
        self.use_json = use_json
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]

        # Define class mapping consistently: 0=weed, 1=maize
        self.class_map = {'weed': 0, 'maize': 1}

    def __len__(self):
        return len(self.image_files)

    def load_annotations_from_json(self, json_path, img_width, img_height):
        try:
            with open(json_path, 'r') as f:
                annotations = json.load(f)

            boxes = []
            labels = []

            for anno in annotations[0]['annotations']:
                x = anno['coordinates']['x']
                y = anno['coordinates']['y']
                width = anno['coordinates']['width']
                height = anno['coordinates']['height']

                # Validate box dimensions
                if width <= 0 or height <= 0:
                    continue

                x_min = max(0, x - width / 2)
                y_min = max(0, y - height / 2)
                x_max = min(img_width, x + width / 2)
                y_max = min(img_height, y + height / 2)

                # Ensure box has positive area
                if x_max <= x_min or y_max <= y_min:
                    continue

                boxes.append([x_min, y_min, x_max, y_max])
                label = self.class_map.get(anno['label'], 0)
                labels.append(label)

            return boxes, labels
        except Exception as e:
            print(f"Error loading JSON annotations from {json_path}: {e}")
            return [], []

    def load_annotations_from_txt(self, txt_path, img_width, img_height):
        try:
            boxes = []
            labels = []

            if not os.path.exists(txt_path):
                print(f"Warning: Annotation file not found: {txt_path}")
                return [], []

            with open(txt_path, 'r') as f:
                lines = f.readlines()

            for line in lines:
                values = line.strip().split()
                if len(values) >= 5:
                    # In TXT format: 0 = maize, 1 = weed (based on example)
                    original_label = int(values[0])
                    # Convert to our consistent mapping: 0=weed, 1=maize
                    label = 1 if original_label == 0 else 0

                    x_center = float(values[1]) * img_width
                    y_center = float(values[2]) * img_height
                    width = float(values[3]) * img_width
                    height = float(values[4]) * img_height

                    # Validate box dimensions
                    if width <= 0 or height <= 0:
                        continue

                    x_min = max(0, x_center - width / 2)
                    y_min = max(0, y_center - height / 2)
                    x_max = min(img_width, x_center + width / 2)
                    y_max = min(img_height, y_center + height / 2)

                    # Ensure box has positive area
                    if x_max <= x_min or y_max <= y_min:
                        continue

                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(label)

            return boxes, labels
        except Exception as e:
            print(f"Error loading TXT annotations from {txt_path}: {e}")
            return [], []

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.root_dir, img_name)

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a blank image with no annotations as fallback
            img = Image.new("RGB", (100, 100), color=(0, 0, 0))
            target = {
                'boxes': torch.zeros((0, 4), dtype=torch.float32),
                'labels': torch.zeros((0,), dtype=torch.int64),
                'image_id': torch.tensor([idx]),
                'area': torch.zeros((0,), dtype=torch.float32),
                'iscrowd': torch.zeros((0,), dtype=torch.int64)
            }
            if self.transform:
                img = self.transform(img)
            return img, target, img_name

        img_width, img_height = img.size

        if self.use_json:
            anno_path = os.path.join(self.root_dir, img_name.replace('.jpg', '.json'))
            boxes, labels = self.load_annotations_from_json(anno_path, img_width, img_height)
        else:
            anno_path = os.path.join(self.root_dir, img_name.replace('.jpg', '.txt'))
            boxes, labels = self.load_annotations_from_txt(anno_path, img_width, img_height)

        # Handle empty annotations
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            'iscrowd': torch.zeros((len(labels),), dtype=torch.int64)
        }

        if self.transform:
            img = self.transform(img)

        return img, target, img_name

def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Initialize bias for better class balance
    with torch.no_grad():
        model.roi_heads.box_predictor.cls_score.bias.data[0] = -2.0  # weed
        model.roi_heads.box_predictor.cls_score.bias.data[1] = 1.0   # maize

    return model

def visualize_prediction(image, prediction, threshold=0.5):
    draw = ImageDraw.Draw(image)
    boxes = prediction['boxes'].cpu().numpy()
    scores = prediction['scores'].cpu().numpy()
    labels = prediction['labels'].cpu().numpy()

    colors = ['red', 'green']  # weed: red, maize: green
    class_names = ['weed', 'maize']

    for box, score, label in zip(boxes, scores, labels):
        if score >= threshold:
            # Convert to int to use as coordinates
            box = box.astype(int)

            # Ensure valid box coordinates
            if box[0] >= box[2] or box[1] >= box[3]:
                continue

            draw.rectangle([(box[0], box[1]), (box[2], box[3])],
                         outline=colors[label], width=3)
            draw.text((box[0], box[1]-10),
                    f"{class_names[label]}: {score:.2f}",
                    fill=colors[label])
    return image

def train_model(model, train_loader, val_loader, optimizer, scheduler, output_dir, num_epochs=15):
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    model.to(device)

    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    best_model_path = os.path.join(output_dir, 'best_model.pth')

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')

        # Training phase
        model.train()
        train_loss = 0.0

        for images, targets, _ in tqdm(train_loader, desc="Training"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            optimizer.zero_grad()

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            # Check for invalid loss
            if not torch.isfinite(losses):
                print(f"WARNING: Non-finite loss detected, skipping batch")
                continue

            losses.backward()

            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_loss += losses.item()

        # Validation phase - FIXED: Use model.eval() during validation
        val_loss = 0.0
        model.train()  # Set model to evaluation mode
        with torch.no_grad():
            for images, targets, _ in tqdm(val_loader, desc="Validating"):
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                loss_dict = model(images, targets)
                val_loss += sum(loss for loss in loss_dict.values()).item()

        # Calculate epoch metrics
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['lr'].append(current_lr)

        print(f'Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr:.6f}')

        # Learning rate scheduling - FIXED: Properly use ReduceLROnPlateau
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(avg_val_loss)
            else:
                scheduler.step()

        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            print(f'New best model saved with val loss: {best_val_loss:.4f}')

    # Plot training history
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss')

    plt.subplot(1, 2, 2)
    plt.plot(history['lr'])
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.title('Learning Rate Schedule')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_history.png'))
    plt.close()

    return model, history

def calculate_metrics(model, data_loader, iou_threshold=0.5, score_threshold=0.5):
    """Calculate precision, recall, and F1 for the model on the given dataset."""
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    model.to(device)
    model.eval()

    class_metrics = {
        0: {'tp': 0, 'fp': 0, 'fn': 0, 'gt': 0},  # weed
        1: {'tp': 0, 'fp': 0, 'fn': 0, 'gt': 0}   # maize
    }

    with torch.no_grad():
        for images, targets, _ in tqdm(data_loader, desc="Calculating metrics"):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for i, (output, target) in enumerate(zip(outputs, targets)):
                target_boxes = target['boxes'].cpu().numpy()
                target_labels = target['labels'].cpu().numpy()

                pred_boxes = output['boxes'].cpu().numpy()
                pred_scores = output['scores'].cpu().numpy()
                pred_labels = output['labels'].cpu().numpy()

                # Filter predictions by score threshold
                mask = pred_scores >= score_threshold
                pred_boxes = pred_boxes[mask]
                pred_scores = pred_scores[mask]
                pred_labels = pred_labels[mask]

                # Count ground truth for each class
                for label in target_labels:
                    class_metrics[label.item()]['gt'] += 1

                # Mark all ground truth as not matched yet
                matched = [False] * len(target_boxes)

                # Sort predictions by confidence score (high to low)
                sort_idx = np.argsort(pred_scores)[::-1]
                pred_boxes = pred_boxes[sort_idx]
                pred_labels = pred_labels[sort_idx]

                # For each prediction, find the best matching ground truth
                for pred_box, pred_label in zip(pred_boxes, pred_labels):
                    best_iou = iou_threshold
                    best_gt_idx = -1

                    for gt_idx, (gt_box, gt_label) in enumerate(zip(target_boxes, target_labels)):
                        if matched[gt_idx] or gt_label != pred_label:
                            continue

                        # Calculate IoU
                        xmin = max(gt_box[0], pred_box[0])
                        ymin = max(gt_box[1], pred_box[1])
                        xmax = min(gt_box[2], pred_box[2])
                        ymax = min(gt_box[3], pred_box[3])

                        if xmax <= xmin or ymax <= ymin:
                            continue

                        intersection = (xmax - xmin) * (ymax - ymin)
                        gt_area = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])
                        pred_area = (pred_box[2] - pred_box[0]) * (pred_box[3] - pred_box[1])
                        union = gt_area + pred_area - intersection

                        iou = intersection / union

                        if iou > best_iou:
                            best_iou = iou
                            best_gt_idx = gt_idx

                    if best_gt_idx >= 0:
                        # True positive
                        matched[best_gt_idx] = True
                        class_metrics[pred_label]['tp'] += 1
                    else:
                        # False positive
                        class_metrics[pred_label]['fp'] += 1

                # Count false negatives
                for gt_idx, (matched_gt, gt_label) in enumerate(zip(matched, target_labels)):
                    if not matched_gt:
                        class_metrics[gt_label.item()]['fn'] += 1

    # Calculate metrics for each class
    results = {}
    for class_id, metrics in class_metrics.items():
        if metrics['tp'] + metrics['fp'] > 0:
            precision = metrics['tp'] / (metrics['tp'] + metrics['fp'])
        else:
            precision = 0

        if metrics['tp'] + metrics['fn'] > 0:
            recall = metrics['tp'] / (metrics['tp'] + metrics['fn'])
        else:
            recall = 0

        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0

        results[class_id] = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': metrics['gt']
        }

    # Calculate average metrics
    avg_precision = sum(m['precision'] for m in results.values()) / len(results)
    avg_recall = sum(m['recall'] for m in results.values()) / len(results)
    avg_f1 = sum(m['f1'] for m in results.values()) / len(results)

    results['average'] = {
        'precision': avg_precision,
        'recall': avg_recall,
        'f1': avg_f1
    }

    return results

def main():
    data_dir = "/kaggle/input/project-2/Annotated-Maize-Weed-Images"
    output_dir = "/kaggle/working/results"
    os.makedirs(output_dir, exist_ok=True)

    # Improved data augmentation for better generalization
    train_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.2))
    ])

    # No augmentation for validation/testing
    test_transform = transforms.Compose([
        transforms.ToTensor()
    ])

    # Create dataset
    dataset = MaizeWeedDataset(data_dir, transform=train_transform, use_json=True)

    # Split dataset
    train_size = int(0.7 * len(dataset))
    val_size = int(0.15 * len(dataset))
    test_size = len(dataset) - train_size - val_size

    # Use fixed seed for reproducibility
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [train_size, val_size, test_size],
        generator=generator
    )

    # Apply different transforms to validation and test sets
    val_dataset.dataset.transform = test_transform
    test_dataset.dataset.transform = test_transform

    # Create data loaders with better performance
    train_loader = DataLoader(
        train_dataset,
        batch_size=4,
        shuffle=True,
        collate_fn=lambda x: tuple(zip(*x)),
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=4,
        collate_fn=lambda x: tuple(zip(*x)),
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        collate_fn=lambda x: tuple(zip(*x)),
        num_workers=2,
        pin_memory=True
    )

    # Initialize model
    model = get_model(num_classes=2)  # 2 classes: weed and maize

    # Better optimizer configuration
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4,
        amsgrad=True
    )

    # Improved learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=2,
        verbose=True,
        min_lr=1e-6
    )

    # Train model
    print("Starting training...")
    model, history = train_model(model, train_loader, val_loader, optimizer, scheduler, output_dir, num_epochs=15)

    # Load best model for evaluation (fixed path)
    best_model_path = os.path.join(output_dir, 'best_model.pth')
    model.load_state_dict(torch.load(best_model_path))
    model.eval()

    # Calculate metrics on test set (new functionality)
    print("\nEvaluating on test set...")
    test_metrics = calculate_metrics(model, test_loader)

    # Print metrics
    print("\nTest Set Metrics:")
    print(f"Average Precision: {test_metrics['average']['precision']:.4f}")
    print(f"Average Recall: {test_metrics['average']['recall']:.4f}")
    print(f"Average F1 Score: {test_metrics['average']['f1']:.4f}")

    print("\nClass-wise Metrics:")
    print(f"Weed - Precision: {test_metrics[0]['precision']:.4f}, Recall: {test_metrics[0]['recall']:.4f}")
    print(f"Maize - Precision: {test_metrics[1]['precision']:.4f}, Recall: {test_metrics[1]['recall']:.4f}")

    # Save metrics to file
    with open(os.path.join(output_dir, 'test_metrics.json'), 'w') as f:
        json.dump(test_metrics, f, indent=4)

    # Run inference and save visualizations
    for images, _, img_names in tqdm(test_loader, desc="Processing test images"):
        image = images[0].to(torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
        with torch.no_grad():
            prediction = model([image])[0]

        original_image = transforms.ToPILImage()(images[0].cpu())
        result_image = visualize_prediction(original_image, prediction)
        result_image.save(os.path.join(output_dir, f"pred_{img_names[0]}"))

    print(f"\nTraining and evaluation complete. Results saved to {output_dir}")

if __name__ == "__main__":
    main()


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
100%|██████████| 167M/167M [00:01<00:00, 144MB/s]  
/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting training...
Epoch 1/15


Validating: 100%|██████████| 19/19 [00:06<00:00,  2.72it/s]


Train Loss: 1.0534 | Val Loss: 0.9898 | LR: 0.001000
New best model saved with val loss: 0.9898
Epoch 2/15


Validating: 100%|██████████| 19/19 [00:06<00:00,  2.76it/s]


...


Evaluating on test set...


Calculating metrics: 100%|██████████| 75/75 [00:09<00:00,  8.18it/s]



Test Set Metrics:
Average Precision: 0.4196
Average Recall: 0.4862
Average F1 Score: 0.4505

Class-wise Metrics:
Weed - Precision: 0.0000, Recall: 0.0000
Maize - Precision: 0.8393, Recall: 0.9724


Processing test images: 100%|██████████| 75/75 [00:13<00:00,  5.54it/s]


Training and evaluation complete. Results saved to /kaggle/working/results


In [ ]:
# def show_predictions(model, dataset, num_images=3, threshold=0.7):
#     device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
#     model.to(device)
#     model.eval()

#     fig, axes = plt.subplots(nrows=1, ncols=num_images, figsize=(20, 5))

#     for idx in range(num_images):
#         # Get random image from dataset
#         random_idx = np.random.randint(0, len(dataset))
#         img, target, img_name = dataset[random_idx]

#         # Prepare image for model
#         img_tensor = img.to(device)

#         # Make prediction
#         with torch.no_grad():
#             prediction = model([img_tensor])[0]

#         # Convert tensor image back to PIL
#         original_image = transforms.ToPILImage()(img.cpu())

#         # Create figure and plot
#         ax = axes[idx] if num_images > 1 else axes
#         ax.imshow(original_image)
#         ax.axis('off')
#         ax.set_title(f"Predictions: {img_name}")

#         # Plot predictions
#         boxes = prediction['boxes'].cpu().numpy()
#         scores = prediction['scores'].cpu().numpy()
#         labels = prediction['labels'].cpu().numpy()

#         # Filter by confidence threshold
#         valid_detections = scores >= threshold

#         # Create custom colormap: red for weed (0), green for maize (1)
#         colors = ['red', 'green']
#         class_names = ['Weed', 'Maize']

#         for box, score, label in zip(boxes[valid_detections],
#                                    scores[valid_detections],
#                                    labels[valid_detections]):
#             # Convert coordinates to image size
#             xmin, ymin, xmax, ymax = box
#             width = xmax - xmin
#             height = ymax - ymin

#             # Create rectangle patch
#             rect = patches.Rectangle(
#                 (xmin, ymin), width, height,
#                 linewidth=2,
#                 edgecolor=colors[label],
#                 facecolor='none'
#             )

#             # Add text label
#             text = f"{class_names[label]}: {score:.2f}"
#             ax.text(
#                 xmin, ymin-10, text,
#                 color=colors[label],
#                 fontsize=10,
#                 bbox=dict(facecolor='white', alpha=0.7, edgecolor='none')
#             )

#             ax.add_patch(rect)

#     plt.tight_layout()
#     plt.show()




In [ ]:
# import os
# import json
# import torch
# import torch.utils.data as data
# from torch.utils.data import DataLoader, random_split
# import torchvision
# from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
# from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
# import torchvision.transforms as transforms
# from torchvision.models.detection.anchor_utils import AnchorGenerator
# from PIL import Image, ImageDraw
# import numpy as np
# from tqdm import tqdm
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches
# # Config - Set these first!
# DATA_DIR = "/kaggle/input/project-2/Annotated-Maize-Weed-Images"
# OUTPUT_DIR = "/kaggle/working/results"
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# NUM_EPOCHS = 15
# BATCH_SIZE = 4

In [ ]:
# class MaizeWeedDataset(data.Dataset):
#     def __init__(self, root_dir, transform=None, use_json=True):
#         self.root_dir = root_dir
#         self.transform = transform
#         self.use_json = use_json
#         self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]

#         # Define class mapping consistently: 0=weed, 1=maize
#         self.class_map = {'weed': 1, 'maize': 0}

#     def __len__(self):
#         return len(self.image_files)

#     def load_annotations_from_json(self, json_path, img_width, img_height):
#         try:
#             with open(json_path, 'r') as f:
#                 annotations = json.load(f)

#             boxes = []
#             labels = []

#             for anno in annotations[0]['annotations']:
#                 x = anno['coordinates']['x']
#                 y = anno['coordinates']['y']
#                 width = anno['coordinates']['width']
#                 height = anno['coordinates']['height']

#                 # Validate box dimensions
#                 if width <= 0 or height <= 0:
#                     continue

#                 x_min = max(0, x - width / 2)
#                 y_min = max(0, y - height / 2)
#                 x_max = min(img_width, x + width / 2)
#                 y_max = min(img_height, y + height / 2)

#                 # Ensure box has positive area
#                 if x_max <= x_min or y_max <= y_min:
#                     continue

#                 boxes.append([x_min, y_min, x_max, y_max])
#                 label = self.class_map.get(anno['label'], 0)
#                 labels.append(label)

#             return boxes, labels
#         except Exception as e:
#             print(f"Error loading JSON annotations from {json_path}: {e}")
#             return [], []

#     def load_annotations_from_txt(self, txt_path, img_width, img_height):
#         try:
#             boxes = []
#             labels = []

#             if not os.path.exists(txt_path):
#                 print(f"Warning: Annotation file not found: {txt_path}")
#                 return [], []

#             with open(txt_path, 'r') as f:
#                 lines = f.readlines()

#             for line in lines:
#                 values = line.strip().split()
#                 if len(values) >= 5:
#                     # In TXT format: 0 = maize, 1 = weed (based on example)
#                     original_label = int(values[0])
#                     # Convert to our consistent mapping: 0=weed, 1=maize
#                     label = 1 if original_label == 0 else 0

#                     x_center = float(values[1]) * img_width
#                     y_center = float(values[2]) * img_height
#                     width = float(values[3]) * img_width
#                     height = float(values[4]) * img_height

#                     # Validate box dimensions
#                     if width <= 0 or height <= 0:
#                         continue

#                     x_min = max(0, x_center - width / 2)
#                     y_min = max(0, y_center - height / 2)
#                     x_max = min(img_width, x_center + width / 2)
#                     y_max = min(img_height, y_center + height / 2)

#                     # Ensure box has positive area
#                     if x_max <= x_min or y_max <= y_min:
#                         continue

#                     boxes.append([x_min, y_min, x_max, y_max])
#                     labels.append(label)

#             return boxes, labels
#         except Exception as e:
#             print(f"Error loading TXT annotations from {txt_path}: {e}")
#             return [], []

#     def __getitem__(self, idx):
#         img_name = self.image_files[idx]
#         img_path = os.path.join(self.root_dir, img_name)

#         try:
#             img = Image.open(img_path).convert("RGB")
#         except Exception as e:
#             print(f"Error loading image {img_path}: {e}")
#             # Return a blank image with no annotations as fallback
#             img = Image.new("RGB", (100, 100), color=(0, 0, 0))
#             target = {
#                 'boxes': torch.zeros((0, 4), dtype=torch.float32),
#                 'labels': torch.zeros((0,), dtype=torch.int64),
#                 'image_id': torch.tensor([idx]),
#                 'area': torch.zeros((0,), dtype=torch.float32),
#                 'iscrowd': torch.zeros((0,), dtype=torch.int64)
#             }
#             if self.transform:
#                 img = self.transform(img)
#             return img, target, img_name

#         img_width, img_height = img.size

#         if self.use_json:
#             anno_path = os.path.join(self.root_dir, img_name.replace('.jpg', '.json'))
#             boxes, labels = self.load_annotations_from_json(anno_path, img_width, img_height)
#         else:
#             anno_path = os.path.join(self.root_dir, img_name.replace('.jpg', '.txt'))
#             boxes, labels = self.load_annotations_from_txt(anno_path, img_width, img_height)

#         # Handle empty annotations
#         if len(boxes) == 0:
#             boxes = torch.zeros((0, 4), dtype=torch.float32)
#             labels = torch.zeros((0,), dtype=torch.int64)
#         else:
#             boxes = torch.as_tensor(boxes, dtype=torch.float32)
#             labels = torch.as_tensor(labels, dtype=torch.int64)

#         target = {
#             'boxes': boxes,
#             'labels': labels,
#             'image_id': torch.tensor([idx]),
#             'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
#             'iscrowd': torch.zeros((len(labels),), dtype=torch.int64)
#         }

#         if self.transform:
#             img = self.transform(img)

#         return img, target, img_name

# def get_model(num_classes):
#     model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')

#     # Agricultural-optimized anchors
#     anchor_sizes = (
#         (32, 64),         # P3: Small weeds
#         (64, 128),        # P4: Medium plants
#         (128, 256),       # P5: Mature maize
#         (256, 512),       # P6: Weed clusters
#         (512,)            # P7: Large groups
#     )
#     aspect_ratios = ((0.8, 1.0, 1.5),) * len(anchor_sizes)  # Plant-oriented ratios

#     model.rpn.anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)

#     # Keep classifier modification
#     in_features = model.roi_heads.box_predictor.cls_score.in_features
#     model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

#     return model




# def visualize_prediction(image, prediction, threshold=0.5):
#     draw = ImageDraw.Draw(image)
#     boxes = prediction['boxes'].cpu().numpy()
#     scores = prediction['scores'].cpu().numpy()
#     labels = prediction['labels'].cpu().numpy()
#     weed_threshold = 0.5  # Lower threshold for weeds
#     maize_threshold = 0.7
#     colors = ['red', 'green']  # weed: red, maize: green
#     class_names = ['weed', 'maize']

#     for box, score, label in zip(boxes, scores, labels):
#         threshold = weed_threshold if label == 0 else maize_threshold
#         if score >= threshold:
#             # Convert to int to use as coordinates
#             box = box.astype(int)

#             # Ensure valid box coordinates
#             if box[0] >= box[2] or box[1] >= box[3]:
#                 continue

#             draw.rectangle([(box[0], box[1]), (box[2], box[3])],
#                          outline=colors[label], width=3)
#             draw.text((box[0], box[1]-10),
#                     f"{class_names[label]}: {score:.2f}",
#                     fill=colors[label])
#     return image

In [ ]:
# def train_model(model, train_loader, val_loader, optimizer, scheduler, output_dir, num_epochs=15):
#     device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
#     model.to(device)

#     # Class weights for loss calculation (weed: 2.0, maize: 1.0)
#     class_weights = torch.tensor([2.0, 1.0]).to(device)

#     best_val_loss = float('inf')
#     history = {
#         'train_loss': [],
#         'val_loss': [],
#         'train_weed_recall': [],
#         'train_maize_recall': [],
#         'lr': []
#     }

#     for epoch in range(num_epochs):
#         print(f'Epoch {epoch+1}/{num_epochs}')

#         # Training Phase
#         model.train()
#         epoch_train_loss = 0.0
#         weed_stats = {'tp': 0, 'fp': 0, 'fn': 0}
#         maize_stats = {'tp': 0, 'fp': 0, 'fn': 0}

#         for batch_idx, (images, targets, _) in enumerate(tqdm(train_loader, desc="Training")):
#             images = [img.to(device) for img in images]
#             targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

#             optimizer.zero_grad()

#             # Forward pass with both losses
#             loss_dict = model(images, targets)

#             # Class-balanced loss calculation
#             losses = sum(
#                 loss * class_weights[target['labels']].mean() if key == 'loss_classifier'
#                 else loss
#                 for key, loss in loss_dict.items()
#             )

#             if not torch.isfinite(losses):
#                 print(f"Invalid loss detected at batch {batch_idx}, skipping update")
#                 continue

#             losses.backward()
#             torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#             optimizer.step()

#             epoch_train_loss += losses.item()

#             # Track class-specific metrics
#             with torch.no_grad():
#                 for target in targets:
#                     labels = target['labels'].cpu().numpy()
#                     weed_count = np.sum(labels == 0)
#                     maize_count = np.sum(labels == 1)
#                     weed_stats['fn'] += weed_count
#                     maize_stats['fn'] += maize_count

#         # Validation Phase
#         model.train()  # Critical: stay in train mode for loss calculation
#         epoch_val_loss = 0.0
#         with torch.no_grad():
#             for images, targets, _ in tqdm(val_loader, desc="Validating"):
#                 images = [img.to(device) for img in images]
#                 targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

#                 val_loss_dict = model(images, targets)
#                 val_losses = sum(
#                     loss * class_weights[target['labels']].mean() if key == 'loss_classifier'
#                     else loss
#                     for key, loss in val_loss_dict.items()
#                 )
#                 epoch_val_loss += val_losses.item()

#         # Calculate metrics
#         avg_train_loss = epoch_train_loss / len(train_loader)
#         avg_val_loss = epoch_val_loss / len(val_loader)
#         current_lr = optimizer.param_groups[0]['lr']

#         # Calculate recall for training data
#         weed_recall = weed_stats['tp'] / (weed_stats['tp'] + weed_stats['fn'] + 1e-7)
#         maize_recall = maize_stats['tp'] / (maize_stats['tp'] + maize_stats['fn'] + 1e-7)

#         history['train_loss'].append(avg_train_loss)
#         history['val_loss'].append(avg_val_loss)
#         history['train_weed_recall'].append(weed_recall)
#         history['train_maize_recall'].append(maize_recall)
#         history['lr'].append(current_lr)

#         print(f'Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')
#         print(f'Weed Recall: {weed_recall:.2%} | Maize Recall: {maize_recall:.2%}')
#         print(f'Learning Rate: {current_lr:.2e}')

#         # Learning rate scheduling
#         if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
#             scheduler.step(avg_val_loss)
#         elif scheduler is not None:
#             scheduler.step()

#         # Save best model
#         if avg_val_loss < best_val_loss:
#             best_val_loss = avg_val_loss
#             torch.save(model.state_dict(), os.path.join(output_dir, 'best_model.pth'))
#             print(f'New best model saved with val loss: {best_val_loss:.4f}')

#     # Plot training history
#     plt.figure(figsize=(15, 5))

#     plt.subplot(1, 3, 1)
#     plt.plot(history['train_loss'], label='Train')
#     plt.plot(history['val_loss'], label='Validation')
#     plt.title('Loss Curve')
#     plt.xlabel('Epoch')
#     plt.legend()

#     plt.subplot(1, 3, 2)
#     plt.plot(history['train_weed_recall'], label='Weed')
#     plt.plot(history['train_maize_recall'], label='Maize')
#     plt.title('Training Recall')
#     plt.xlabel('Epoch')
#     plt.legend()

#     plt.subplot(1, 3, 3)
#     plt.plot(history['lr'])
#     plt.title('Learning Rate')
#     plt.xlabel('Epoch')
#     plt.ylabel('LR')

#     plt.tight_layout()
#     plt.savefig(os.path.join(output_dir, 'training_report.png'))
#     plt.close()

#     return model, history


In [ ]:
# # Load best model
# best_model_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
# model.load_state_dict(torch.load(best_model_path,
#                                map_location=torch.device('cpu'),
#                                weights_only=True))  # Fix security warning

# # Evaluate
# test_metrics = calculate_metrics(model, test_loader)
# print("Weed Metrics:", test_metrics[1])  # Now using 1 for weed
# print("Maize Metrics:", test_metrics[0])

# # Visualize predictions
# show_predictions(model, test_dataset.dataset)


In [ ]:
# def main():
#     # Configuration
#     data_dir = "/kaggle/input/project-2/Annotated-Maize-Weed-Images"
#     output_dir = "/kaggle/working/results"
#     os.makedirs(output_dir, exist_ok=True)

#     # Hyperparameters
#     config = {
#         "batch_size": 4,
#         "num_epochs": 15,
#         "learning_rate": 0.001,
#         "weight_decay": 0.0001,
#         "lr_patience": 3,
#         "min_lr": 1e-6
#     }

#     # Data Preparation ---------------------------------------------------------
#     print("Preparing datasets...")

#     # Custom transforms with enhanced augmentation for weeds
#     train_transform = transforms.Compose([
#         transforms.ToTensor(),
#         transforms.RandomHorizontalFlip(p=0.5),
#         transforms.RandomVerticalFlip(p=0.3),
#         transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
#         transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
#     ])

#     val_transform = transforms.Compose([
#         transforms.ToTensor()
#     ])

#     # Create datasets
#     full_dataset = MaizeWeedDataset(data_dir, transform=train_transform, use_json=True)

#     # Split dataset (70% train, 15% val, 15% test)
#     train_size = int(0.7 * len(full_dataset))
#     val_size = int(0.15 * len(full_dataset))
#     test_size = len(full_dataset) - train_size - val_size

#     train_dataset, val_dataset, test_dataset = random_split(
#         full_dataset, [train_size, val_size, test_size],
#         generator=torch.Generator().manual_seed(42)
#     )

#     # Apply different transforms to validation set
#     val_dataset.dataset.transform = val_transform
#     test_dataset.dataset.transform = val_transform

#     # Create data loaders
#     train_loader = DataLoader(
#         train_dataset,
#         batch_size=config['batch_size'],
#         shuffle=True,
#         num_workers=2,
#         collate_fn=lambda x: tuple(zip(*x))
#     )

#     val_loader = DataLoader(
#         val_dataset,
#         batch_size=config['batch_size'],
#         shuffle=False,
#         num_workers=2,
#         collate_fn=lambda x: tuple(zip(*x))
#     )

#     test_loader = DataLoader(
#         test_dataset,
#         batch_size=1,  # Use batch_size=1 for evaluation
#         shuffle=False,
#         num_workers=2,
#         collate_fn=lambda x: tuple(zip(*x))
#     )

#     # Model Setup --------------------------------------------------------------
#     print("Initializing model...")
#     model = get_model(num_classes=2)

#     # Optimizer with weight decay
#     optimizer = torch.optim.AdamW(
#         model.parameters(),
#         lr=config['learning_rate'],
#         weight_decay=config['weight_decay']
#     )

#     # Learning rate scheduler
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer,
#         mode='min',
#         factor=0.5,
#         patience=config['lr_patience'],
#         min_lr=config['min_lr'],
#         verbose=True
#     )

#     # Training ----------------------------------------------------------------
#     print("Starting training...")
#     model, history = train_model(
#         model,
#         train_loader,
#         val_loader,
#         optimizer,
#         scheduler,
#         output_dir,
#         num_epochs=config['num_epochs']
#     )

#     # Evaluation --------------------------------------------------------------
#     print("\nEvaluating on test set...")

#     # Load best model weights
#     best_model_path = os.path.join(output_dir, 'best_model.pth')
#     model.load_state_dict(torch.load(best_model_path, map_location=device))
#     model.eval()

#     # Calculate metrics
#     test_metrics = calculate_metrics(model, test_loader)

#     print("\nTest Set Metrics:")
#     print(f"Average Precision: {test_metrics['average']['precision']:.4f}")
#     print(f"Average Recall: {test_metrics['average']['recall']:.4f}")
#     print(f"Average F1 Score: {test_metrics['average']['f1']:.4f}\n")

#     print("Class-wise Metrics:")
#     print(f"Weed - Precision: {test_metrics[0]['precision']:.4f}, Recall: {test_metrics[0]['recall']:.4f}")
#     print(f"Maize - Precision: {test_metrics[1]['precision']:.4f}, Recall: {test_metrics[1]['recall']:.4f}")

#     # Visualize predictions
#     print("\nGenerating sample predictions...")
#     show_predictions(model, test_dataset.dataset, num_images=3)

#     print(f"\nTraining and evaluation complete. Results saved to {output_dir}")

In [ ]:
# if __name__ == "__main__":
#     # Detect hardware
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     print(f"Using device: {device}")

#     # Run pipeline
#     main()

In [ ]:
# import os
# import json
# import torch
# from PIL import Image
# import torch.utils.data as data
# import torchvision.transforms as transforms
# from torch.utils.data import DataLoader, random_split
# import glob

# class MaizeWeedDataset(data.Dataset):
#     def __init__(self, root_dir, transform=None):
#         self.root_dir = root_dir
#         self.transform = transform
#         self.img_paths = sorted(glob.glob(os.path.join(root_dir, "*.jpg")))

#     def __len__(self):
#         return len(self.img_paths)

#     def __getitem__(self, idx):
#         img_path = self.img_paths[idx]
#         image = Image.open(img_path).convert("RGB")

#         # Get annotations from JSON file
#         json_path = img_path.replace('.jpg', '.json')
#         with open(json_path, 'r') as f:
#             json_data = json.load(f)[0]

#         boxes = []
#         labels = []

#         for ann in json_data['annotations']:
#             # Extract coordinates (center format)
#             x = ann['coordinates']['x']
#             y = ann['coordinates']['y']
#             width = ann['coordinates']['width']
#             height = ann['coordinates']['height']

#             # Convert to [x_min, y_min, x_max, y_max] format
#             x_min = max(0, x - width / 2)
#             y_min = max(0, y - height / 2)
#             x_max = x + width / 2
#             y_max = y + height / 2

#             boxes.append([x_min, y_min, x_max, y_max])
#             # Convert label to numeric: 'maize' -> 0, 'weed' -> 1
#             labels.append(0 if ann['label'] == 'maize' else 1)

#         # Convert to tensors
#         boxes = torch.as_tensor(boxes, dtype=torch.float32)
#         labels = torch.as_tensor(labels, dtype=torch.int64)

#         # Create target dictionary
#         target = {
#             'boxes': boxes,
#             'labels': labels,
#             'image_id': torch.tensor([idx]),
#             'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
#             'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64)
#         }

#         if self.transform:
#             image = self.transform(image)

#         return image, target
# #

In [ ]:
# from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
# from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
# from torchvision.models.detection.anchor_utils import AnchorGenerator

# def get_model(num_classes=2):
#     # Initialize with default weights
#     model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')

#     # Configure anchors properly for FPN levels
#     anchor_sizes = (
#         (32, 64),         # P3: Small objects
#         (64, 128),        # P4: Medium objects
#         (128, 256),       # P5: Large objects
#         (256, 512),       # P6: Larger groups
#         (512,)            # P7: Field-wide context
#     )
#     aspect_ratios = ((0.8, 1.0, 1.5),) * len(anchor_sizes)

#     # Replace anchor generator
#     model.rpn.anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)

#     # Important: Reduce proposal counts to avoid memory issues
#     model.rpn.pre_nms_top_n = {'training': 2000, 'testing': 1000}
#     model.rpn.post_nms_top_n = {'training': 1000, 'testing': 500}

#     # Modify classifier head for our classes
#     in_features = model.roi_heads.box_predictor.cls_score.in_features
#     model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

#     return model


In [ ]:
# from tqdm import tqdm

# def train_model(model, train_loader, val_loader=None, optimizer=None, scheduler=None,
#                 output_dir="./", num_epochs=15):
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     model.to(device)

#     if optimizer is None:
#         # Optimizer with weight decay
#         params = [p for p in model.parameters() if p.requires_grad]
#         optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

#     if scheduler is None:
#         # Learning rate scheduler
#         scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

#     history = {'train_loss': [], 'val_loss': []}

#     for epoch in range(num_epochs):
#         print(f"Epoch {epoch+1}/{num_epochs}")

#         # Training phase
#         model.train()
#         epoch_loss = 0

#         for images, targets in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
#             # Move to device
#             images = list(image.to(device) for image in images)
#             targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

#             # Zero the gradients
#             optimizer.zero_grad()

#             # Forward pass with both losses
#             loss_dict = model(images, targets)

#             # Class-balanced loss calculation
#             losses = sum(loss for loss in loss_dict.values())

#             # Backward pass
#             losses.backward()

#             # Clip gradients to prevent explosion
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

#             # Update weights
#             optimizer.step()

#             epoch_loss += losses.item()

#         # Average training loss
#         avg_loss = epoch_loss / len(train_loader)
#         history['train_loss'].append(avg_loss)
#         print(f"Training Loss: {avg_loss:.4f}")

#         # Validation phase
#         if val_loader:
#             model.eval()
#             val_loss = 0

#             with torch.no_grad():
#                 for images, targets in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}"):
#                     images = list(image.to(device) for image in images)
#                     targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

#                     # Compute validation loss
#                     loss_dict = model(images, targets)
#                     losses = sum(loss for loss in loss_dict.values())
#                     val_loss += losses.item()

#             # Average validation loss
#             avg_val_loss = val_loss / len(val_loader)
#             history['val_loss'].append(avg_val_loss)
#             print(f"Validation Loss: {avg_val_loss:.4f}")

#         # Update learning rate
#         scheduler.step()

#         # Save checkpoint
#         if (epoch + 1) % 5 == 0:
#             torch.save({
#                 'epoch': epoch,
#                 'model_state_dict': model.state_dict(),
#                 'optimizer_state_dict': optimizer.state_dict(),
#                 'loss': avg_loss,
#             }, f"{output_dir}/model_epoch_{epoch+1}.pth")

#     return model, history


In [ ]:
# def main():
#     # Set paths
#     data_path = "/kaggle/input/project-2/Annotated-Maize-Weed-Images"
#     output_dir = "./"

#     # Check device
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     print(f"Using device: {device}")

#     # Data preparation
#     print("Preparing datasets...")
#     transform = transforms.Compose([
#         transforms.Resize((800, 800)),  # Resize to manageable dimensions
#         transforms.ToTensor(),
#         transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
#     ])

#     # Create dataset and split
#     dataset = MaizeWeedDataset(root_dir=data_path, transform=transform)
#     train_size = int(0.8 * len(dataset))
#     val_size = len(dataset) - train_size
#     train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

#     # Create data loaders with custom collate function
#     def collate_fn(batch):
#         return tuple(zip(*batch))

#     train_loader = DataLoader(
#         train_dataset, batch_size=4, shuffle=True,
#         num_workers=2, collate_fn=collate_fn
#     )

#     val_loader = DataLoader(
#         val_dataset, batch_size=4, shuffle=False,
#         num_workers=2, collate_fn=collate_fn
#     )

#     # Model initialization
#     print("Initializing model...")
#     model = get_model(num_classes=2)  # 0: maize, 1: weed

#     # Optimizer with class weights
#     params = [p for p in model.parameters() if p.requires_grad]
#     optimizer = torch.optim.SGD(params, lr=0.003, momentum=0.9, weight_decay=0.0005)

#     # LR scheduler with warmup
#     scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

#     # Training
#     print("Starting training...")
#     model, history = train_model(
#         model=model,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         optimizer=optimizer,
#         scheduler=scheduler,
#         output_dir=output_dir,
#         num_epochs=15
#     )

#     # Save final model
#     torch.save(model.state_dict(), f"{output_dir}/final_model.pth")
#     print("Training complete!")

# if __name__ == "__main__":
#     main()
